In [1]:
# !pip install openai==0.28

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 798.0 kB/s eta 0:00:00


In [14]:
# ##uncomment this to use code in Google Colab
# !pip install fastchat
# !pip install wget
# !pip install evaluate
# !pip install sentence_transformers
# !pip install bert_score
# !pip install datasets

In [15]:
# ##uncomment this to use code in Google Colab
# from IPython.display import clear_output
# !git clone https://github.com/alfekka/lm-polygraph.git -b new_api
# %cd lm-polygraph
# %pip install .
# %cd src
# %pip install transformers rouge-score datasets openai
# !curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
# !apt-get install -y nodejs
# %cd lm_polygraph/app
# !npm install
# %cd ../../
# %cd /content/lm-polygraph/src
# %load_ext autoreload
# %autoreload 2

Cloning into 'lm-polygraph'...
remote: Enumerating objects: 6842, done.
remote: Counting objects: 100% (2952/2952), done.
remote: Compressing objects: 100% (781/781), done.
remote: Total 6842 (delta 2394), reused 2311 (delta 2081), pack-reused 3890
Receiving objects: 100% (6842/6842), 13.25 MiB | 12.80 MiB/s, done.
Resolving deltas: 100% (4096/4096), done.
/content/lm-polygraph/src/lm-polygraph
Processing /content/lm-polygraph/src/lm-polygraph
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 30.1 MB/s eta 0:00:00
  Using cached nlpaug-1.1.11-py3-none-any.whl (410 kB)
  Using cached bs4-0.0.2-py2.py3-none-any.whl (1.2 kB)
  Using cached transformers-4.36.0-py3-none-any.whl (8.2 MB)
  Using cached sacrebleu-2.4.2-py3-none-any.whl (106 kB)
  Using cached flask-3.0.3-py3-none-any.whl (101 kB)
  Using cach

In [16]:
# ##uncomment this to use code in Google Colab
# %cd ..
# !pwd
# %cd /content/lm-polygraph/src

/content/lm-polygraph
/content/lm-polygraph
/content/lm-polygraph/src


In [17]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [18]:
import os
from lm_polygraph.stat_calculators import *
from lm_polygraph.utils.openai_chat import OpenAIChat
from transformers import AutoTokenizer
from lm_polygraph.model_adapters import WhiteBoxModelBasic
from torch.utils.data import DataLoader
import json
import pandas as pd
from tqdm import tqdm
## Create a ClaimsExtractor object
calc_claim_extractor = ClaimsExtractor(OpenAIChat("gpt-4o"))
## Set model to None, as we will not need it
model = None
tokenizer = AutoTokenizer.from_pretrained("Vikhrmodels/Vikhr-7B-instruct_0.3", use_fast=False, trust_remote_code=True)
## Setting a dummy whitebox model object
model_adapter = WhiteBoxModelBasic(model, tokenizer)


In [20]:
from datasets import load_dataset
full = load_dataset("rvanova/person-bio-part")
russian_bios = pd.DataFrame(full['test'])

In [22]:
bios_with_claims = []
## Create a DataLoader object
for  index, row in tqdm(russian_bios.iterrows(), total=russian_bios.shape[0]):

    question = row['input_texts']
    bio = row['bio_ru']
    print(bio)


    encoded = tokenizer([bio], padding=True, return_tensors="pt")
    deps = {"greedy_tokens": encoded.input_ids, "greedy_texts": [bio]}
    claims = calc_claim_extractor(deps, texts=bio, model=model_adapter,language="ru")

    for x in claims['claims'][0]:
        bios_with_claims.append({'question': question, 'bio': bio, "claim": x.claim_text, "sentence": x.sentence, "tokens": x.aligned_tokens})


  0%|          | 0/80 [00:00<?, ?it/s]

\n bot\nИван Петрович Павлов (1849-1936) - русский физиолог, лауреат Нобелевской премии по физиологии и медицине 1904 года.\n\nБиография Ивана Петровича Павлова:\n\n1. Родился 26 сентября (8 октября) 1849 года в Рязани, в семье священника.\n2. В 1864 году поступил в Рязанскую духовную семинарию, но через год был исключен за неуспеваемость.\n3. В 1869 году поступил в Московский университет на медицинский факультет.\n4. В 1875 году окончил университет и получил степень доктора медицины.\n5. В 1876 году начал работать в лаборатории И. М. Сеченова, где занимался изучением физиологии пищеварения.\n6. В 1890 году стал профессором физиологии в Военно-медицинской академии в Санкт-Петербурге.\n7. В 1897 году открыл условные рефлексы, за что в 1904 году получил Нобелевскую премию по физиологии и медицине.\n8. В 1918 году стал президентом Российской


  1%|▏         | 1/80 [00:39<52:04, 39.54s/it]

\n bot\nМихаил Васильевич Ломоносов (1711-1765) - русский ученый, поэт, художник, историк, просветитель и государственный деятель. Родился 8 (19) ноября 1711 года в деревне Мишанинской (ныне село Ломоносово) Архангельской губернии в семье крестьянина-помора.\n\nВ 1730 году Ломоносов отправился в Москву, где поступил в Славяно-греко-латинскую академию. В 1735 году он был отправлен в Петербург для продолжения обучения в Академической гимназии. В 1736 году Ломоносов был отправлен в Германию для продолжения образования.\n\nВ 1741 году Ломоносов вернулся в Россию и стал профессором химии и металлургии в Академии наук. В 1745 году он был назначен адъюнктом Академии наук, а в 1755 году - профессором химии и металлургии.\n\nВ 1756 году Ломоносов основал Московский университет, который стал первым российским университетом. В 1760 году он был назначен профессором химии и металлургии в


  2%|▎         | 2/80 [01:12<46:44, 35.96s/it]

...

 98%|█████████▊| 78/80 [45:36<01:14, 37.38s/it]

. bot\n\nАндрей Николаевич Туполев (1888-1972) - советский авиаконструктор, академик АН СССР, Герой Социалистического Труда, генерал-полковник инженерно-технической службы, лауреат Ленинской и шести Сталинских премий.\n\nБиография Андрея Туполева:\n\n1. Родился 10 ноября 1888 года в городе Торопец, Российская империя.\n2. В 1908 году окончил Московское высшее техническое училище (ныне МГТУ им. Н. Э. Баумана) по специальности "инженер-механик".\n3. В 1918 году начал работать в Центральном аэрогидродинамическом институте (ЦАГИ) в Москве.\n4. В 1922 году стал руководителем конструкторского бюро ЦАГИ.\n5. В 1932 году был назначен главным конструктором авиационного завода No 156 в Москве.\n6. В 1936 году был назначен главным конструктором авиационного завода No 156 в Москве.\n7. В 1937 году был арестован по обвинению в ш


 99%|█████████▉| 79/80 [46:19<00:39, 39.28s/it]

\n bot\nАлександр Степанович Попов (1859-1905) - русский физик и электротехник, изобретатель радио. Родился 16 (4) марта 1859 года в городе Турьинские Рудники (ныне Краснотурьинск) на Урале.\n\nВ 1873 году Попов поступил в Петербургский университет, где изучал физику и математику. В 1882 году он окончил университет и начал работать преподавателем физики в Минном офицерском классе в Кронштадте. В 1883 году Попов начал исследования в области электромагнитных волн и в 1895 году изобрел радиоприемник, который использовал для передачи и приема электромагнитных волн.\n\nВ 1899 году Попов продемонстрировал свое изобретение на заседании Русского физико-химического общества, где он впервые передал радиосигнал на расстояние 250 метров. В 1901 году Попов продемонстрировал свое изобретение на Международной электротехнической выставке в Париже, где он получил золотую медаль.\n\nВ 1905 году Попов умер от


100%|██████████| 80/80 [46:54<00:00, 35.18s/it]


In [23]:
## Save the extracted claims

save_df = pd.DataFrame(bios_with_claims, )

save_df.to_csv("../workdir/russian_claims.csv", index=False)